In [44]:
import requests
import pandas as pd
import datetime

In [45]:
# Load Combined Set Data
combined_set_data = pd.read_csv("../Data/combined_set_data.csv")

In [46]:
def set_pull(): 
    '''
    Pulls the latest card set information from the YGOProDeck API and returns a DataFrame with the relevant data.
    '''
    url = "https://db.ygoprodeck.com/api/v7/cardinfo.php"
    data = requests.get(url).json()["data"]

    rows = []
    for c in data:
        for s in c.get("card_sets", []) or []:
            rows.append({
                "card_id": c["id"],
                "name": c["name"],
                "set_name": s["set_name"],
                "set_code": s["set_code"],
                "rarity": s["set_rarity"],
                "rarity_code": s.get("set_rarity_code"),
                "price": float(s["set_price"]) if s["set_price"] not in (None, "", "0.00") else None
            })

    df = pd.DataFrame(rows)
    df['time_pulled'] = datetime.datetime.now()

    # Save the first set_code for each card_id+set_name+rarity combination
    first_set_code = df.groupby(['card_id', 'set_name', 'rarity'])['set_code'].first().reset_index()

    # Remove the set_code column
    df_nosetcode = df.drop(columns=['set_code'])

    # Remove duplicate rows based on card_id, set_name, and rarity
    df_unique = df_nosetcode.drop_duplicates(subset=['card_id', 'set_name', 'rarity'])

    # Reattach the saved first set_code value
    df_final = df_unique.merge(first_set_code, on=['card_id', 'set_name', 'rarity'], how='left')

    # Calculate number of printings (card sets) for each card_id
    printings_per_card = df_final.groupby('card_id').size().rename('num_printings')

    # Calculate number of unique rarities for each card_id
    rarities_per_card = df_final.groupby('card_id')['rarity'].nunique().rename('num_rarities')

    # Attach these to the main DataFrame
    df_final = df_final.merge(printings_per_card, left_on='card_id', right_index=True)
    df_final = df_final.merge(rarities_per_card, left_on='card_id', right_index=True)
    
    return df_final


In [47]:
df = set_pull()

# Save to Auto Set Pulls named with the current date and time
current_time = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
df.to_csv(f"../Data/Auto Set Pulls/ygo_set_daily_{current_time}.csv", index=False)

In [48]:
# Combine with existing data
combined_set_data = pd.concat([combined_set_data, df], ignore_index=True)

# Update the combined set data CSV
combined_set_data.to_csv("../Data/combined_set_data.csv", index=False)